# <font style="font-family:roboto;color:#455e6c"> Parametrising a Machine Learning Interatomic Potential </font>  

<div class="admonition note" name="html-admonition" style="background:#e3f2fd; padding: 10px">
<font style="font-family:roboto;color:#455e6c"> <b> DPG Tutorial: Automated Workflows and Machine Learning for Materials Science Simulations </b> </font> </br>
<font style="font-family:roboto;color:#455e6c"> 16 March 2025 </font>
</div>

In this notebook we fit an [Atomic Cluster Expansion](https://doi.org/10.1103/PhysRevB.99.014104) interatomic potential using the [pacemaker](https://www.nature.com/articles/s41524-021-00559-9) software.

In [1]:
from pyiron_core import PyironFlow, Workflow
import matplotlib.pyplot as plt

/cmmc/ptmp/pyironhb/mambaforge/envs/pyiron_mpie_cmti_2026-01-19/lib/python3.12/site-packages/nglview/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [10]:
import pyiron_nodes as pn

In [2]:
import pyiron_nodes.atomistic.ml_potentials.fitting.linearfit as lf

In [3]:
workflow_name = "linear_fit"
file_path = "data/mgca.pckl.tgz"
training_frac = 0.5
number_of_functions_per_element = 10
rcut = 6.0
compression = None

wf = Workflow(workflow_name)

# Workflow connections
wf.load_dataset = lf.ReadPickledDatasetAsDataframe(
    file_path=file_path, compression=compression
)
wf.split_dataset = lf.SplitTrainingAndTesting(
    data_df=wf.load_dataset.outputs.df, training_frac=training_frac
)
wf.parameterize_potential = lf.ParameterizePotentialConfig(
    number_of_functions_per_element=number_of_functions_per_element, rcut=rcut
)
wf.run_linear_fit = lf.RunLinearFit(
    potential_config=wf.parameterize_potential,
    df_train=wf.split_dataset.outputs.df_training,
    df_test=wf.split_dataset.outputs.df_testing,
    verbose=False,
)
wf.save_potential = lf.SavePotential(basis=wf.run_linear_fit.outputs.basis)
wf.predict_energies_forces = lf.PredictEnergiesAndForces(
    basis=wf.run_linear_fit.outputs.basis,
    df_train=wf.split_dataset.outputs.df_training,
    df_test=wf.split_dataset.outputs.df_testing,
)

In [4]:
wf.run()

Restoring node outputs  c670ddfd833db04804f4d8a46590407978dc726afaece1c8eb5379359fd72a35 run_linear_fit True
Potentials "Ca_Mg_linear_potential.yaml" and "Ca_Mg_linear_potential.yace" are saved in "/cmmc/u/hgaafer/pyiron/projects/2026_03_DPG_workshop_dresden/DPG-tutorial-2025/Linear_ace_potentials".
Restoring node outputs  ae4f6a754793e5ff9e503ed4b65df7be3fdfc8a0d52e0d654d4daa704f952959 predict_energies_forces False
No stored data found for node:  predict_energies_forces
serialization not needed


{'reference_training_epa': array([-0.68663657, -0.99923799, -1.08362327, ..., -1.47605157,
        -1.44900232, -1.2044471 ], shape=(19340,)),
 'reference_training_fpa': array([-0.0000e+00, -0.0000e+00,  0.0000e+00, ...,  5.5500e-06,
         5.2099e-04,  3.0750e-05], shape=(322638,)),
 'predicted_training_epa': array([-0.81386625, -0.87184381, -0.43534825, ..., -1.51306876,
        -1.50692894, -1.07993029], shape=(19340,)),
 'predicted_training_fpa': array([ 6.66133815e-16,  6.66133815e-16, -1.81972493e-15, ...,
         3.04603864e-05,  5.53148274e-04,  5.84272215e-05], shape=(322638,)),
 'reference_testing_epa': array([-1.50104811,  0.17800853, -0.57766618, ..., -0.83208959,
        -0.25764739, -0.27296486], shape=(19341,)),
 'reference_testing_fpa': array([ 0.        ,  0.        ,  0.        , ..., -0.05524943,
        -0.03189827, -1.23912384], shape=(321417,)),
 'predicted_testing_epa': array([-1.22525141,  0.06423183, -0.70620421, ..., -0.70818788,
        -0.10829411, -0.206

In [5]:
pf = PyironFlow(wf_list = [wf], nodes_path="./pyiron_nodes")
pf.gui

## <font style="font-family:roboto;color:#455e6c"> Loading the dataset </font> 

Recalling the workflow, we are in the first essential step of loading the training dataset

<img src="img/highlighted_workflow.png" width="50%">

Firstly, we load the dataset by providing the `file_path`

In [ ]:
load_dataset = ReadPickledDatasetAsDataframe(file_path = "data/mgca.pckl.tgz", compression = None)
load_dataset.pull();

Using the `PlotEnergyHistogram` Node, we can plot the energy histogram

In [ ]:
hist_plot = PlotEnergyHistogram(df = load_dataset.outputs.df, log_scale = False)
hist_plot.pull();

Similarly, using the `PlotForcesHistogram` Node, we can plot the forces histogram

In [ ]:
hist_plot = PlotForcesHistogram(df = load_dataset.outputs.df, log_scale = True)
hist_plot.pull();

## <font style="font-family:roboto;color:#455e6c"> Split the dataset into training and test </font> 

Secondly, we split the dataset into training and testing datasets.

This is done by choosing the percentage used for training dataset through the `training_frac` parameter where `training_frac = 0.5` means we are using 50% for training and 50% for testing.

In [ ]:
split_dataset = SplitTrainingAndTesting(data_df = load_dataset.outputs.df, training_frac = 0.5)

## <font style="font-family:roboto;color:#455e6c"> Constructing potential configuration </font> 

Similarly as in `PACEmaker`, we want to have a file which is similar to `input.yaml` file that looks like that (Check the `PACEmaker` [documentation](https://pacemaker.readthedocs.io/en/latest/) for more details),

```
fit:
  fit_cycles: 1
  loss: {L1_coeffs: 1e-08, L2_coeffs: 1e-08, kappa: 0.08, w0_rad: 0, w1_rad: 0, w2_rad: 0}
  maxiter: 1000
  optimizer: BFGS
  trainable_parameters: ALL
  weighting: {DE: 1.0, DElow: 1.0, DEup: 10.0, DF: 1.0, DFup: 50.0, energy: convex_hull,
    nfit: 20000, reftype: all, seed: 42, type: EnergyBasedWeightingPolicy, wlow: 0.95}
.
.
.
potential:
  bonds:
    ALL:
      dcut: 0.01
      radbase: SBessel
      radparameters: [5.25]
      rcut: 6.0
  elements: [Mg, Ca]
  embeddings:
    ALL:
      fs_parameters: [1, 1, 1, 0.5]
      ndensity: 2
      npot: FinnisSinclairShiftedScaled
  functions:
    number_of_functions_per_element = 300
    ALL:
      lmax_by_orders: [15, 6, 2, 1]
      nradmax_by_orders: [0, 6, 3, 1]

### 1. Embeddings
 i.e. how atomic energy $E_i$ depends on ACE properties/densities $\varphi$. Linear expansion $E_i = \varphi$ is the trivial. Non-linear expansion, i.e. those, containing square root, gives more flexiblity and accuracy of final potential

Embeddings for `ALL` species: 
- non-linear `FinnisSinclairShiftedScaled`
- 2 densities
- fs_parameters': [1, 1, 1, 0.5]:
$$E_i = 1.0 * \varphi(1)^1 + 1.0 * \varphi(2)^{0.5} = \varphi^{(1)} + \sqrt{\varphi^{(2)}} $$

### 2. Radial functions

Radial functions are orthogonal polynoms example:
* (a) Exponentially-scaled Chebyshev polynomials (λ = 5.25)
* (b) Power-law scaled Chebyshev polynomials (λ = 2.0)
* (c) Simplified spherical Bessel functions

<img src="img/radial-functions-low.png" width="40%">

Radial functions specification for `ALL` species pairs (i.e. Al-Al, Al-Li, Li-Al, Li-Li):

* based on the Simplified Bessel
* cutoff $r_c=6.0$

#### 3. B-basis functions

B-basis functions  for `ALL` species type interactions, i.e. Al-Al block:
* maximum order = 4, i.e. body-order 5 (1 central atom + 4 neighbour  densities)
* nradmax_by_orders: 15, 3, 2, 1
* lmax_by_orders: 0, 3, 2, 1

For simplicity, the main inputs that we will consider for the potential configurations are,

- `number_of_functions_per_element`: specifies how many functions will be provided in the potential
- `rcut`: specifies what the cutoff radius is

In [ ]:
parameterize_potential = ParameterizePotentialConfig(number_of_functions = 10, rcut = 6.0)

Check the current potential configurations in dictionary format,

In [ ]:
parameterize_potential.pull().to_dict()

## <font style="font-family:roboto;color:#455e6c"> Linear fitting </font>

Finally, we run our fit with the `potential_config` we obtained and then save the potential files inside a new folder.

**Note:** Setting `verbose = True` will show all the details of building the design matrices.

In [ ]:
run_linear_fit = RunLinearFit(potential_config = parameterize_potential,
                                                df_train = split_dataset.outputs.df_training,
                                                df_test= split_dataset.outputs.df_testing, verbose = False)

In [ ]:
save_potential = SavePotential(basis = run_linear_fit.outputs.basis)

In [ ]:
save_potential.pull()

## <font style="font-family:roboto;color:#455e6c"> Workflow </font>

In [ ]:
wf = make_linearfit(workflow_name= 'LinearAceDataset', file_path='data/mgca.pckl.tgz', delete_existing_savefiles=True)

In [ ]:
wf.draw(size=(20,10))

You can save the workflow using `wf.save()` to call it back later without the need to re-run it.

In [ ]:
wf.save()

## <font style="font-family:roboto;color:#455e6c"> Loading the workflow in GUI </font>

We want to run the workflow using the GUI and do level 1 validation to our potential.

Helpful node:
- `atomistic -> mlips -> fitting -> ace`: contains all nodes to run the linear ACE fit and to plot the data histograms and the fitting's accuracy curves

<img src="img/validation_schematic.png" width="60%">

**NOTE:** You can change the ratio of the canvas to the whole screen by changing the value of `flow_widget_ratio` between 0 to 1 (try 0.6 or 0.7)

In [ ]:
pf = PyironFlow([wf], flow_widget_ratio = 0.7)
pf.gui

Exercise: change the `number_of_functions_per_element` to a higher value (i.e., 50) and check the fitting curves. Did the fit get better or worse?

**Note:** You can save the potential under a new name using the `filename` input in `save_potential` node.